# Periodic electrodes

An extraction cell of slit grids, one aperture pitch wide, solved by the panel method with a period along $y$, a Neumann face behind the plasma grid and a plate as the far face, instead of stacked and mirrored copies of the electrodes.

The cell repeats every `pitch` along $y$. The plasma grid (0 V) and the extraction grid (5 kV) are one plate per period each, between two half apertures. On the left, $x = 0$ is a symmetry plane: a Neumann `Face` mirrors every charge with its sign. On the right, the far face at 6 kV is an open plate spanning one period, which the period turns into an infinite plane. The panels are graded: short at the aperture tips and longer away from them.

In [ ]:
import time

import numpy as np
import plasma_plots
from simgeometry import Polygon, Polyline

from tree_d_3d import BoundaryIntegralElectrodes, Conductor, Face, conductors_from_shape

plasma_plots.set_backend("plotly")

mm = 1e-3
pitch, half_aperture, thickness = 2.0 * mm, 0.4 * mm, 0.3 * mm
grids = {"plasma grid": (1.0 * mm, 0.0), "extraction grid": (2.5 * mm, 5e3)}
far_face, far_potential = 4.0 * mm, 6e3


def plate(x):
    """The plate of a slit grid at ``x`` between the apertures at y = 0 and y = pitch."""
    lower, upper = half_aperture, pitch - half_aperture
    return Polygon(
        vertices=(
            (x - thickness / 2, lower),
            (x + thickness / 2, lower),
            (x + thickness / 2, upper),
            (x - thickness / 2, upper),
        )
    )


tips = np.array(
    [
        (x + side * thickness / 2, y)
        for x, _ in grids.values()
        for side in (-1, 1)
        for y in (half_aperture, pitch - half_aperture)
    ]
).T


def panel_length(points):
    """10 um at the tips, growing by 0.1 per unit distance, at most 0.1 mm."""
    distance = np.min(np.hypot(*(points[:, :, None] - tips[:, None, :])), axis=1)
    return np.minimum(10e-6 + 0.1 * distance, 0.1 * mm)


conductors = [
    conductor
    for x, potential in grids.values()
    for conductor in conductors_from_shape(plate(x), potential, panel_length)
]
wall = Polyline(vertices=((far_face, 0.0), (far_face, pitch)))
conductors += conductors_from_shape(wall, far_potential, panel_length)

start = time.perf_counter()
cell = BoundaryIntegralElectrodes(conductors, period=pitch, face=Face("x", 0.0))
cell.solve()
print(f"{cell.num_panels} panels, solved in {time.perf_counter() - start:.2f} s")

The potential over two periods: the cell repeats along $y$, and its field is tangential to the Neumann face at $x = 0$.

In [ ]:
x = np.linspace(0.0, far_face, 161)
y = np.linspace(-pitch / 2, 3 * pitch / 2, 161)
table = cell.tabulate((x[0], y[0]), (x[-1], y[-1]), (x.size, y.size))

with plasma_plots.figure(1, 1) as fig:
    ax = fig[0]
    ax.contourf(x / mm, y / mm, table.potential.T / 1e3, levels=30)
    for shift in (-pitch, 0.0, pitch):
        for xg, _ in grids.values():
            outline = np.asarray(plate(xg).vertices)
            corners = np.vstack((outline, outline[:1])) + [0.0, shift]
            ax.plot(corners[:, 0] / mm, corners[:, 1] / mm, "k-")
    ax.set_xlabel("x (mm)")
    ax.set_ylabel("y (mm)")
    ax.set_ylim(y[0] / mm, y[-1] / mm)
    ax.set_title("potential (kV)")
fig

On the axis of an aperture ($y = 0$) the potential rises from the plasma grid to the extraction grid and on to the far face; the field along the axis vanishes at the Neumann face.

In [ ]:
axis = np.stack((x, np.zeros_like(x)))
phi, field = cell.probe(axis)
print(f"E_x at the Neumann face: {field[0, 0]:.1e} V/m")
with plasma_plots.figure(1, 2) as fig:
    fig[0].plot(x / mm, phi / 1e3)
    fig[0].set_xlabel("x (mm)")
    fig[0].set_ylabel("potential on the axis (kV)")
    fig[1].plot(x / mm, -field[0] / 1e6)
    fig[1].set_xlabel("x (mm)")
    fig[1].set_ylabel("$-E_x$ on the axis (MV/m)")
fig

Without the period, the cell has to be repeated by hand, `copies` periods to either side, and mirrored about $x = 0$. The truncated stack leaves an error in the field, largest away from the electrodes, while the dense system grows with every copy.

In [ ]:
probes = np.stack((np.full(5, 1.75 * mm), np.linspace(0.0, pitch, 5, endpoint=False)))
probes = np.concatenate(
    (probes, tips + [[0.03 * mm], [0.0]] * np.sign(tips[0] - 1.75 * mm)), axis=1
)
reference = cell.field(probes)


def mirrored(conductor):
    return Conductor(
        conductor.vertices[:, ::-1] * [[-1.0], [1.0]],
        conductor.potential,
        conductor.closed,
    )


for copies in (1, 2, 4):
    stack = [
        Conductor(c.vertices + [[0.0], [k * pitch]], c.potential, c.closed)
        for k in range(-copies, copies + 1)
        for c in conductors
    ]
    stack += [mirrored(c) for c in stack]
    start = time.perf_counter()
    explicit = BoundaryIntegralElectrodes(stack)
    explicit.solve()
    seconds = time.perf_counter() - start
    error = np.max(np.hypot(*(explicit.field(probes) - reference))) / np.max(
        np.hypot(*reference)
    )
    print(
        f"{copies} copies: {explicit.num_panels:5d} panels, {seconds:5.2f} s, "
        f"largest field error {error:.1e}"
    )